# Lab 05 — Pipeline parallelism by hand + the asymmetry experiment

**Goal:** split GPT-2 into two pipeline stages on one GPU, simulate the network with the **detach trick**, verify gradient-exactness against the unsplit model, then quantize the wire in each direction separately and measure which direction is more fragile.

This notebook is a miniature of the entire proposed system.

In [ ]:
!pip -q install transformers matplotlib
import torch, torch.nn.functional as F, matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
model.train(False)

def quantize(x, bits, block=64, stochastic=False):
    if bits >= 16: return x
    shape = x.shape; flat = x.float().flatten()
    pad = (-flat.numel()) % block
    if pad: flat = torch.cat([flat, flat.new_zeros(pad)])
    fb = flat.view(-1, block); qmax = 2**(bits-1)-1
    s = fb.abs().amax(1, keepdim=True)/qmax + 1e-12
    y = fb/s
    q = torch.floor(y + torch.rand_like(y)) if stochastic else torch.round(y)
    return (q.clamp(-qmax,qmax)*s).flatten()[:x.numel()].view(shape).to(x.dtype)

## 1. Cut the model into two stages
Stage 1 = embeddings + blocks 0..5. Stage 2 = blocks 6..11 + final norm + LM head + loss. The tensor between them is **the boundary activation** — the thing your whole proposal compresses.

In [ ]:
CUT = 6
wte, wpe = model.transformer.wte, model.transformer.wpe
blocks, ln_f, head = model.transformer.h, model.transformer.ln_f, model.lm_head

def stage1(ids):
    pos = torch.arange(ids.size(1), device=ids.device)
    h = wte(ids) + wpe(pos)
    for b in blocks[:CUT]:
        h = b(h)[0]
    return h

def stage2(h, ids):
    for b in blocks[CUT:]:
        h = b(h)[0]
    logits = head(ln_f(h))
    return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                           ids[:, 1:].reshape(-1))

ids = tok("Distributed training over slow networks requires careful engineering of every byte.",
          return_tensors="pt").input_ids.to(device)
print("tokens:", ids.shape)

## 2. The detach trick — simulating the wire
`detach()` severs the autograd graph exactly where the network cable would. `requires_grad_()` starts a fresh graph on the 'receiving machine'. `h.backward(g)` injects the arriving activation-gradient to resume the chain rule upstream. **Read this cell until every line is obvious — it IS the system.**

In [ ]:
model.zero_grad()
h = stage1(ids)                       # forward on 'machine 1'
h_wire = h.detach()                   # ---- network: forward direction ----
h_recv = h_wire.clone().requires_grad_()   # arrives on 'machine 2'
loss = stage2(h_recv, ids)
loss.backward()                       # backward through stage 2 only
g_wire = h_recv.grad                  # ---- network: backward direction ----
h.backward(g_wire)                    # resume backward through stage 1
print("staged loss:", loss.item())

## 3. Correctness check — staged must equal unsplit, exactly
If the split is right, every weight gradient matches the ordinary full-model backward to float tolerance.

In [ ]:
staged = {n: p.grad.clone() for n, p in model.named_parameters() if p.grad is not None}
model.zero_grad()
out = model(ids, labels=ids)
out.loss.backward()
ok = all(torch.allclose(staged[n], p.grad, atol=1e-4)
         for n, p in model.named_parameters() if n in staged)
print("loss match:", abs(out.loss.item() - loss.item()) < 1e-4, "   all grads match:", ok)

If both print True, you have implemented correct 2-stage pipeline parallelism (minus the physical network). Everything from here is *your research*: what happens when the wire lossy-compresses.

## 4. The asymmetry experiment — d_fwd vs d_bwd fragility
Reference = exact staged grads. Then two families of runs: quantize **forward only** at b bits, and **backward only** at b bits. Metric: cosine similarity of stage-1 weight gradients vs reference (1.0 = perfect direction).

In [ ]:
def staged_grads(bits_f=16, bits_b=16):
    model.zero_grad()
    h = stage1(ids)
    h_recv = quantize(h.detach(), bits_f).clone().requires_grad_()
    loss = stage2(h_recv, ids)
    loss.backward()
    h.backward(quantize(h_recv.grad, bits_b, stochastic=True))
    g = torch.cat([p.grad.flatten() for n, p in model.named_parameters()
                   if p.grad is not None and (n.startswith("transformer.h.0")
                   or n.startswith("transformer.h.2") or n.startswith("transformer.h.4"))])
    return g.clone()

ref = staged_grads(16, 16)
cos = lambda g: F.cosine_similarity(ref, g, dim=0).item()

bits_axis = [8, 6, 4, 3, 2]
fwd_only  = [cos(staged_grads(b, 16)) for b in bits_axis]
bwd_only  = [cos(staged_grads(16, b)) for b in bits_axis]
for b, f, w in zip(bits_axis, fwd_only, bwd_only):
    print(f"{b}-bit   fwd-only cosine={f:.4f}   bwd-only cosine={w:.4f}")

In [ ]:
plt.figure(figsize=(6,3.5))
plt.plot(bits_axis, fwd_only, marker="o", label="quantize forward only (d_fwd)")
plt.plot(bits_axis, bwd_only, marker="s", label="quantize backward only (d_bwd)")
plt.gca().invert_xaxis()
plt.xlabel("bit-depth on the wire"); plt.ylabel("grad cosine vs exact (stage 1)")
plt.title("Direction asymmetry: the backward cliff arrives earlier")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

**This figure is the empirical heart of the proposal**: at the same bit-depth, corrupting the backward wire damages the training signal more than corrupting the forward wire — so separate d_fwd/d_bwd budgets (direction-asymmetric precision) are justified. Save it.

## Exercises
1. Move CUT to 2 and 10. Does boundary position change the asymmetry?
2. Turn stochastic rounding **off** for the backward wire — how much worse does bwd-only get? (Bias vs variance, visible.)
3. Extend to 4 stages (3 boundaries) with independent bits per boundary — you now have per-link precision, the full decision space of the proposal.
4. Instead of grad cosine, run 30 actual optimizer steps per config on a small dataset and compare loss curves — the metric your final evaluation uses.